In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import cv2
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Aktif İşlem Birimi: {device}")

ROOT_DIR = Path.cwd().parent
df = pd.read_parquet(ROOT_DIR / "data/processed/glcm_features.parquet")

class_names = sorted(df['defect_class'].unique())
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

class SteelDefectDataset(Dataset):
    def __init__(self, metadata_df, split='train'):
        if split == 'train':
            self.data = metadata_df[metadata_df['split'].str.lower() == 'train'].reset_index(drop=True)
        else:
            self.data = metadata_df[metadata_df['split'].str.lower().isin(['val', 'validation', 'test'])].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = cv2.imread(row['file_path'], cv2.IMREAD_GRAYSCALE)
        
        img_tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        img_tensor = (img_tensor - 0.5) / 0.2
        
        label = class_to_idx[row['defect_class']]
        return img_tensor, torch.tensor(label, dtype=torch.long)

train_dataset = SteelDefectDataset(df, split='train')
val_dataset = SteelDefectDataset(df, split='val')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True)
 
sample_x, sample_y = next(iter(train_loader))
print(f"Eğitim Kümesi    : {len(train_dataset)} görsel ({len(train_loader)} batch)")
print(f"Doğrulama Kümesi : {len(val_dataset)} görsel ({len(val_loader)} batch)")
print(f"Batch Şekli (B, C, H, W): {sample_x.shape}")
print(f"Etiket Şekli            : {sample_y.shape}")

In [ ]:
import torch.nn as nn

class SteelDefectCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(SteelDefectCNN, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)  # (Batch, 128)
        x = self.classifier(x)   # (Batch, 6)
        return x

# Modeli GPU'ya Taşı
model = SteelDefectCNN(num_classes=6).to(device)

# Toplam Parametre Sayısını Hesapla
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("CNN Modeli Başarıyla Oluşturuldu ve GPU'ya Aktarıldı!")
print(f"Toplam Eğitilebilir Parametre Sayısı: {total_params:,}")

In [ ]:
import time
import copy
from pathlib import Path

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 15
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f"GPU Üzerinde ({EPOCHS} Epoch)...\n")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    

    model.train()
    train_loss = 0.0
    train_correct = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()            # Önceki adımdan kalan gradyanları sıfırla
        outputs = model(images)          # Forward Pass
        loss = criterion(outputs, labels)# Cross Entropy
        loss.backward()                  # Backpropagation
        optimizer.step()                 # Ağırlıkları Güncelle
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data).item()
        
    epoch_train_loss = train_loss / len(train_dataset)
    epoch_train_acc = train_correct / len(train_dataset)
    

    model.eval()
    val_loss = 0.0
    val_correct = 0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct / len(val_dataset)
    elapsed = time.time() - start_time
    
    # Metrikleri Kaydet
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    # En İyi Modeli Belleğe Al
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        saved_flag = "En İyi Model"
    else:
        saved_flag = ""
        
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.2f}s) | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: %{epoch_train_acc*100:.2f} | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: %{epoch_val_acc*100:.2f} {saved_flag}")

# En İyi Ağırlıkları Yükle ve Diske Kaydet
model.load_state_dict(best_model_wts)
models_dir = ROOT_DIR / "models"
models_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), models_dir / "cnn_baseline.pth")

print(f"\nEğitim Tamamlandı: En Yüksek Doğrulama Başarısı: %{best_val_acc*100:.2f}")
print(f"Model Ağırlıkları Kaydedildi -> {models_dir / 'cnn_baseline.pth'}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
axes[0].plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', color='#d62728', lw=2)
axes[0].set_title('Loss Değişimi (Epochs)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS + 1), [acc * 100 for acc in history['train_acc']], label='Train Acc', color='#1f77b4', lw=2)
axes[1].plot(range(1, EPOCHS + 1), [acc * 100 for acc in history['val_acc']], label='Val Acc', color='#2ca02c', lw=2)
axes[1].set_title('Accuracy Değişimi (%)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

plt.figure(figsize=(8, 6))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'CNN Baseline Confusion Matrix (Acc: %{best_val_acc*100:.2f})', fontsize=12, fontweight='bold')
plt.xlabel('Tahmin Edilen (Predicted)', fontsize=11)
plt.ylabel('Gerçek Sınıf (Actual)', fontsize=11)
plt.tight_layout()
plt.show()

print("--- CNN Baseline Sınıflandırma Raporu ---")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import pandas as pd
import torch

from src.data.dataset import build_dataloaders
from src.models.cnn import SteelDefectCNN, train_cnn_model, evaluate_model

ROOT_DIR = Path.cwd().parent
df = pd.read_parquet(ROOT_DIR / "data/processed/glcm_features.parquet")
train_loader, val_loader, class_to_idx, idx_to_class = build_dataloaders(df, batch_size=32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SteelDefectCNN(num_classes=len(class_to_idx)).to(device)

trained_model, history = train_cnn_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=15,
    save_path=str(ROOT_DIR / "models/cnn_baseline.pth")
)

eval_results = evaluate_model(trained_model, val_loader, device, list(class_to_idx.keys()))